In [187]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, OrdinalEncoder, MaxAbsScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score, recall_score, precision_score, make_scorer
from imblearn import FunctionSampler
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import make_column_selector as selector
import pandas as pd
from sklearn.model_selection import GroupKFold, StratifiedKFold, StratifiedGroupKFold, KFold, cross_validate
from sklearn import set_config
from sklearn.base import clone
import warnings

# Esto hace que los steps de las pipelines sean df de pandas por defecto, sino serian np arrays
set_config(transform_output="pandas")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [188]:
df_claim_reviews = pd.read_csv("../data/claim_reviews.csv")

In [189]:
df_claim_reviews

,review_id,claim_id,review_round,review_datetime,reviewer_id,review_type,auto_risk_score,triage_result,reviewer_notes,perito_id,perito_physical_inspection,damage_consistency_rating,documentation_completeness_pct,recommended_action,days_since_claim
0,REV_00002_1,CLM_00002,1,2025-11-29 14:33:00,R-2512,Initial Triage,16.1,Green - Proceed,"Standard claim, within normal parameters.",NaN,NaN,NaN,NaN,Proceed,1
1,REV_00003_1,CLM_00003,1,2024-06-26 04:36:00,R-1744,Initial Triage,36.3,Yellow - Standard Review,Claimant was vague about accident circumstances.,NaN,NaN,NaN,NaN,Standard Review,2
2,REV_00003_2,CLM_00003,2,2024-07-29 21:36:00,R-1744,Expert Assessment,NaN,NaN,No anomalies found during review.,P-5992,No,2.2,60.2,Recommend Denial,35
3,REV_00003_3,CLM_00003,3,2024-08-15 13:36:00,SIU-209,SIU Investigation,NaN,NaN,Full investigation completed. Findings documen...,P-5992,Yes,NaN,NaN,Claim Denied,51
4,REV_00004_1,CLM_00004,1,2025-08-05 19:55:00,R-3644,Initial Triage,29.8,Green - Proceed,"All documents verified, consistent narrative.",NaN,NaN,NaN,NaN,Proceed,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20888,REV_15413_1,CLM_15413,1,2023-07-06 08:20:00,R-9461,Initial Triage,26.8,Green - Proceed,"Routine processing, no flags raised.",NaN,NaN,NaN,NaN,Proceed,1
20889,REV_15414_1,CLM_15414,1,2025-10-10 01:53:00,R-6221,Initial Triage,67.4,Orange - Escalated Review,Third party account differs slightly from clai...,NaN,NaN,NaN,NaN,Escalated Review,1
20890,REV_15414_2,CLM_15414,2,2025-10-16 13:53:00,R-6221,Expert Assessment,NaN,NaN,Documentation supports claim as reported.,P-8545,Yes,4.3,26.5,Approve with Conditions,7
20891,REV_15417_1,CLM_15417,1,2022-11-10 21:46:00,R-8044,Initial Triage,28.3,Green - Proceed,"Documentation complete, no issues detected.",NaN,NaN,NaN,NaN,Proceed,2


In [190]:
#Ver nulos

df_claim_reviews.isnull().sum()

review_id                             0
claim_id                              0
review_round                          0
review_datetime                       0
reviewer_id                           0
review_type                           0
auto_risk_score                    8626
triage_result                      8626
reviewer_notes                        0
perito_id                         12267
perito_physical_inspection        12267
damage_consistency_rating         13897
documentation_completeness_pct    13897
recommended_action                    0
days_since_claim                      0
dtype: int64

In [191]:
df_claim_reviews.describe(include='all')

,review_id,claim_id,review_round,review_datetime,reviewer_id,review_type,auto_risk_score,triage_result,reviewer_notes,perito_id,perito_physical_inspection,damage_consistency_rating,documentation_completeness_pct,recommended_action,days_since_claim
count,20893,20893,20893.000000,20893,20893,20893,12267.000000,12267,20893,8626,8626,6996.000000,6996.000000,20893,20893.000000
unique,20893,12267,NaN,20784,791,3,NaN,4,40,59,2,NaN,NaN,15,NaN
top,REV_15419_1,CLM_00008,NaN,2025-02-10 08:23:00,R-2207,Initial Triage,NaN,Green - Proceed,"Documentation complete, no issues detected.",P-9976,No,NaN,NaN,Proceed,NaN
freq,1,3,NaN,3,603,12267,NaN,6183,2391,309,4756,NaN,NaN,6183,NaN
mean,NaN,NaN,1.490882,NaN,NaN,NaN,30.992574,NaN,NaN,NaN,NaN,3.606718,80.303902,NaN,7.622218
std,NaN,NaN,0.637157,NaN,NaN,NaN,20.705931,NaN,NaN,NaN,NaN,0.892093,13.696789,NaN,9.188407
min,NaN,NaN,1.000000,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,1.000000,14.700000,NaN,1.000000
25%,NaN,NaN,1.000000,NaN,NaN,NaN,15.200000,NaN,NaN,NaN,NaN,3.000000,71.575000,NaN,2.000000
50%,NaN,NaN,1.000000,NaN,NaN,NaN,29.700000,NaN,NaN,NaN,NaN,3.700000,81.050000,NaN,4.000000
75%,NaN,NaN,2.000000,NaN,NaN,NaN,45.400000,NaN,NaN,NaN,NaN,4.300000,90.700000,NaN,10.000000


CALCULO DEL TARGET

In [192]:
# Condición A: ronda 3 con resolución negativa SIU
cond_a_actions = {
    "Claim Denied",
    "Claim Withdrawn by Claimant",
    "Pending Litigation",
}
cond_a_claims = set(
    df_claim_reviews.loc[
        (df_claim_reviews["review_round"] == 3)
        & (df_claim_reviews["recommended_action"].isin(cond_a_actions)),
        "claim_id",
    ]
)

# Condición B: ronda 2 con acción desfavorable + inconsistencia de daños < 3.0
cond_b_actions = {
    "Recommend Denial",
    "Refer to Special Investigations Unit",
}
cond_b_claims = set(
    df_claim_reviews.loc[
        (df_claim_reviews["review_round"] == 2)
        & (df_claim_reviews["recommended_action"].isin(cond_b_actions))
        & (df_claim_reviews["damage_consistency_rating"] < 3.0),
        "claim_id",
    ]
)

# Condición C: ronda 1 roja + ronda 2 con inspección física y docs < 65
r1_red_claims = set(
    df_claim_reviews.loc[
        (df_claim_reviews["review_round"] == 1)
        & (df_claim_reviews["triage_result"] == "Red - Full Investigation"),
        "claim_id",
    ]
)
r2_severe_claims = set(
    df_claim_reviews.loc[
        (df_claim_reviews["review_round"] == 2)
        & (df_claim_reviews["perito_physical_inspection"] == "Yes")
        & (df_claim_reviews["documentation_completeness_pct"] < 65),
        "claim_id",
    ]
)
cond_c_claims = r1_red_claims & r2_severe_claims

# Unión lógica A OR B OR C
fraud_claims = cond_a_claims | cond_b_claims | cond_c_claims

# Target final por siniestro
df_target = (
    df_claim_reviews[["claim_id"]]
    .drop_duplicates()
    .assign(fraud_flag=lambda d: d["claim_id"].isin(fraud_claims).astype(int))
    .sort_values("claim_id")
    .reset_index(drop=True)
)

print("Nº siniestros:", len(df_target))
print("Fraudes (fraud_flag=1):", int(df_target["fraud_flag"].sum()))
print("No fraudes (fraud_flag=0):", int((df_target["fraud_flag"] == 0).sum()))

df_target.head(10)

Nº siniestros: 12267
Fraudes (fraud_flag=1): 878
No fraudes (fraud_flag=0): 11389


,claim_id,fraud_flag
0,CLM_00002,0
1,CLM_00003,1
2,CLM_00004,0
3,CLM_00005,0
4,CLM_00006,0
5,CLM_00007,0
6,CLM_00008,1
7,CLM_00009,0
8,CLM_00010,0
9,CLM_00012,0


In [193]:
df_target['fraud_flag'].value_counts(normalize=True)

fraud_flag
0    0.928426
1    0.071574
Name: proportion, dtype: float64

In [194]:
df_target

,claim_id,fraud_flag
0,CLM_00002,0
1,CLM_00003,1
2,CLM_00004,0
3,CLM_00005,0
4,CLM_00006,0
...,...,...
12262,CLM_15412,0
12263,CLM_15413,0
12264,CLM_15414,0
12265,CLM_15417,0


### Teniendo ya calculado el target, vamos a abandonar la tabla claim_reviews (en el readme.md se especifica que solo sirve para calcular el target)

In [195]:
df_claims = pd.read_csv("..\\data\\claims.csv")

In [196]:
df_claims

,claim_id,policy_id,customer_id,vehicle_id,agent_id,accident_datetime,claim_datetime,fault,accident_area,accident_description,accident_latitude,accident_longitude,police_report_filed,witness_present,number_of_supplements,claimed_amount_eur,repair_workshop
0,CLM_00002,POL_07423,CUS_01111,VEH_01155,AGT_00118,2025-11-02 16:33:00,2025-11-28 06:33:00,Policy Holder,Urban,flooding damage,41.104921,-1.472463,Yes,Unknown,7,1198.91,Garage Ramírez - Murcia
1,CLM_00003,POL_01086,CUS_01750,VEH_00648,AGT_00027,2024-05-12 22:36:00,2024-06-23 15:36:00,Policy Holder,Urban,glass breakage,37.332902,-1.063439,No,Yes,7,11848.24,Carrocerías Díaz - Coruña
2,CLM_00004,POL_06373,CUS_06806,VEH_00899,AGT_00051,2025-07-09 22:55:00,2025-08-03 02:55:00,Policy Holder,Urban,bicycle involved,36.944930,-6.714446,Yes,No,2,1116.78,Carrocerías Díaz - Vigo
3,CLM_00005,POL_04543,CUS_04165,VEH_05968,AGT_00062,2023-11-08 20:14:00,2023-11-20 17:14:00,Policy Holder,Urban,multi-vehicle pileup,42.466545,2.320600,Yes,Yes,1,2246.88,Carrocerías Ruiz - Valencia
4,CLM_00006,POL_04676,CUS_07070,VEH_06082,AGT_00076,2025-08-21 12:39:00,2025-09-30 18:39:00,Policy Holder,Urban,falling object,39.341821,-2.414222,No,Yes,6,449.41,Carrocerías Gómez - Valencia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12262,CLM_15412,POL_07393,CUS_02209,VEH_07329,AGT_00123,2024-12-17 11:01:00,2024-12-26 16:01:00,Policy Holder,Suburban,pedestrian involved,39.549568,0.227634,Yes,No,0,960.23,Mecánica Rubio - Málaga
12263,CLM_15413,POL_00752,CUS_05111,VEH_02884,AGT_00105,2023-06-24 02:20:00,2023-07-04 19:20:00,Policy Holder,Urban,theft attempt,38.652884,-3.561560,No,No,2,1843.21,Taller Domínguez - Alicante
12264,CLM_15414,POL_04232,CUS_01224,VEH_03091,AGT_00042,2025-09-10 06:53:00,2025-10-08 17:53:00,Policy Holder,Urban,theft attempt,38.648536,-6.392743,No,Yes,2,957.81,Taller Rivera - Alicante
12265,CLM_15417,POL_02434,CUS_01106,VEH_06088,AGT_00173,2022-10-11 14:46:00,2022-11-08 04:46:00,Policy Holder,Urban,weather-related,42.800512,-3.779887,No,No,5,2349.20,Chapa y Pintura Castillo - Las Palmas


In [197]:
df_claims.isnull().sum()

claim_id                 0
policy_id                0
customer_id              0
vehicle_id               0
agent_id                 0
accident_datetime        0
claim_datetime           0
fault                    0
accident_area            0
accident_description     0
accident_latitude        0
accident_longitude       0
police_report_filed      0
witness_present          0
number_of_supplements    0
claimed_amount_eur       0
repair_workshop          0
dtype: int64

In [198]:
df_claims.describe(include='all')

,claim_id,policy_id,customer_id,vehicle_id,agent_id,accident_datetime,claim_datetime,fault,accident_area,accident_description,accident_latitude,accident_longitude,police_report_filed,witness_present,number_of_supplements,claimed_amount_eur,repair_workshop
count,12267,12267,12267,12267,12267,12267,12267,12267,12267,12267,12267.000000,12267.000000,12267,12267,12267.000000,12267.000000,12267
unique,12267,6167,3759,6594,180,12227,12228,2,5,20,NaN,NaN,3,3,NaN,NaN,387
top,CLM_15419,POL_00734,CUS_00250,VEH_08192,AGT_00016,2025-05-31 09:23:00,2024-01-11 16:56:00,Policy Holder,Urban,fire damage,NaN,NaN,No,No,NaN,NaN,Carrocerías Ramírez - Vitoria
freq,1,7,19,7,96,2,2,8017,4879,654,NaN,NaN,6684,7194,NaN,NaN,80
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,39.850914,-2.970615,NaN,NaN,3.477541,2999.131380,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.252370,3.624874,NaN,NaN,2.279481,3873.780393,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36.000737,-9.299434,NaN,NaN,0.000000,47.330000,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.889615,-6.085252,NaN,NaN,1.000000,912.225000,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,39.833320,-2.989663,NaN,NaN,3.000000,1822.520000,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,41.788675,0.189406,NaN,NaN,5.000000,3555.105000,NaN


In [199]:
df_customers = pd.read_csv("..\\data\\customers.csv")

In [200]:
df_customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5760 entries, 0 to 5759
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               5760 non-null   object
 1   full_name                 5760 non-null   object
 2   sex                       5760 non-null   object
 3   marital_status            5760 non-null   object
 4   date_of_birth             5760 non-null   object
 5   email                     5760 non-null   object
 6   phone                     5760 non-null   object
 7   address                   5760 non-null   object
 8   city                      5760 non-null   object
 9   province                  5760 non-null   object
 10  postal_code               5760 non-null   int64 
 11  last_address_change_date  3099 non-null   object
dtypes: int64(1), object(11)
memory usage: 540.1+ KB


In [201]:
# Miro nulos por qué en la última columna parece que hay
df_customers.isnull().sum()

customer_id                    0
full_name                      0
sex                            0
marital_status                 0
date_of_birth                  0
email                          0
phone                          0
address                        0
city                           0
province                       0
postal_code                    0
last_address_change_date    2661
dtype: int64

In [202]:
df_customers.describe(include='all')

,customer_id,full_name,sex,marital_status,date_of_birth,email,phone,address,city,province,postal_code,last_address_change_date
count,5760,5760,5760,5760,5760,5760,5760,5760,5760,5760,5760.000000,3099
unique,5760,5502,2,4,5089,5760,5759,5572,40,29,NaN,1271
top,CUS_07179,Manuel Morales Jiménez,Male,Married,1983-11-26,andres.vazquez628@telefonica.net,+34 616 755 616,105 Cypress Blvd,Córdoba,Madrid,NaN,2025-03-31
freq,1,3,3142,2524,4,1,2,4,164,863,NaN,11
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27175.066493,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14944.518719,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1002.000000,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14241.500000,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27420.500000,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40186.000000,NaN


# Columna last_address_change_date
Los nulos que hay en last_address_change_date son por que no se han cambiado de dirección, por lo que lo cambio por No Change
Hacer una columna sustituyendo a last_address_change_date que sea Si o No han cambiado
Borrarla
# Columnas de address, city, province y postal_code
Decidir cuáles borrar o cuál quedarse, por qué por ejemplo postal code no tiene mucho sentido teniendo la ciudad

In [203]:
df_policies = pd.read_csv("..\\data\\policies.csv")
df_vehicles = pd.read_csv("..\\data\\vehicles.csv")

In [204]:
df_policies

,policy_id,customer_id,policy_type,base_policy,deductible,annual_premium_eur,policy_start_date,policy_end_date,past_number_of_claims,number_of_cars
0,POL_00001,CUS_03475,Sport - Collision,Collision,300,1584.19,2022-04-22,2023-04-22,none,7
1,POL_00002,CUS_02564,Sport - Liability,Liability,500,499.00,2021-01-05,2022-01-05,more than 4,1
2,POL_00003,CUS_03003,Utility - Liability,Liability,700,1678.45,2024-04-06,2025-04-06,none,3
3,POL_00004,CUS_02879,Sedan - Collision,Collision,300,1161.30,2018-10-03,2019-10-03,1,1
4,POL_00005,CUS_00266,Utility - All Perils,All Perils,700,525.80,2020-10-31,2021-10-31,more than 4,1
...,...,...,...,...,...,...,...,...,...,...
7611,POL_09493,CUS_05458,Sport - All Perils,All Perils,500,1023.03,2024-05-17,2025-05-17,2,2
7612,POL_09496,CUS_03334,Sport - Collision,Collision,700,911.71,2024-03-23,2025-03-23,1,1
7613,POL_09497,CUS_00802,Sedan - Liability,Liability,700,835.37,2022-08-15,2023-08-15,none,1
7614,POL_09498,CUS_04069,Sedan - Collision,Collision,300,3157.89,2021-03-29,2022-03-29,4,2


In [205]:
df_policies.describe(include='all')

,policy_id,customer_id,policy_type,base_policy,deductible,annual_premium_eur,policy_start_date,policy_end_date,past_number_of_claims,number_of_cars
count,7616,7616,7616,7616,7616.000000,7616.000000,7616,7616,7616,7616.000000
unique,7616,4213,9,3,NaN,NaN,2373,2373,6,NaN
top,POL_09477,CUS_03024,Sport - Liability,Liability,NaN,NaN,2020-10-31,2021-10-31,none,NaN
freq,1,8,888,2596,NaN,NaN,10,10,2244,NaN
mean,NaN,NaN,NaN,NaN,452.166492,752.425376,NaN,NaN,NaN,4.477810
std,NaN,NaN,NaN,NaN,129.995329,400.533312,NaN,NaN,NaN,2.275169
min,NaN,NaN,NaN,NaN,300.000000,85.300000,NaN,NaN,NaN,1.000000
25%,NaN,NaN,NaN,NaN,400.000000,475.567500,NaN,NaN,NaN,3.000000
50%,NaN,NaN,NaN,NaN,400.000000,666.870000,NaN,NaN,NaN,4.000000
75%,NaN,NaN,NaN,NaN,500.000000,927.937500,NaN,NaN,NaN,6.000000


In [206]:
df_vehicles

,vehicle_id,policy_id,license_plate,make,model,manufacture_year,vehicle_category,purchase_price_eur,color,odometer_km,driver_rating
0,VEH_00000,POL_05817,3486 XMR,Toyota,Hilux,2024,Utility,17476.28,Negro,203751,1
1,VEH_00001,POL_06074,9916 KVH,Volvo,XC60,2013,Utility,17966.90,Blanco,118504,3
2,VEH_00002,POL_09056,2787 NWJ,Jaguar,F-Pace,2024,Coupe,10593.51,Blanco,220197,2
3,VEH_00003,POL_06022,6635 CLH,SEAT,Arona,2013,Sedan,36615.40,Blanco,92432,4
4,VEH_00005,POL_06071,6905 HFG,Kia,Rio,2005,SUV,18241.39,Gris,206190,1
...,...,...,...,...,...,...,...,...,...,...,...
6999,VEH_08793,POL_03097,8977 XTB,Nissan,Leaf,2021,Sedan,70410.92,Plata,241831,2
7000,VEH_08794,POL_01633,4119 NJY,Citroën,C5 Aircross,2009,SUV,12993.44,Plata,247804,2
7001,VEH_08795,POL_07575,9016 JMB,Citroën,C4,2015,SUV,8973.63,Negro,21684,3
7002,VEH_08798,POL_04575,0864 YNM,Citroën,C5 Aircross,2018,Sedan,12148.37,Gris,89800,2


In [207]:
df_vehicles.describe(include='all')

,vehicle_id,policy_id,license_plate,make,model,manufacture_year,vehicle_category,purchase_price_eur,color,odometer_km,driver_rating
count,7004,7004,7004,7004,7004,7004.000000,7004,7004.000000,7004,7004.000000,7004.000000
unique,7004,4604,7004,30,172,NaN,6,NaN,11,NaN,NaN
top,VEH_08799,POL_00540,3781 CJH,Volkswagen,Berlingo,NaN,Sedan,NaN,Blanco,NaN,NaN
freq,1,6,1,258,65,NaN,2457,NaN,1370,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,2014.564963,NaN,21688.400938,NaN,140282.830240,2.172616
std,NaN,NaN,NaN,NaN,NaN,5.821509,NaN,14283.256374,NaN,80398.564593,0.965809
min,NaN,NaN,NaN,NaN,NaN,2005.000000,NaN,2094.690000,NaN,506.000000,1.000000
25%,NaN,NaN,NaN,NaN,NaN,2009.000000,NaN,11988.257500,NaN,70006.500000,1.000000
50%,NaN,NaN,NaN,NaN,NaN,2015.000000,NaN,17963.130000,NaN,138858.000000,2.000000
75%,NaN,NaN,NaN,NaN,NaN,2020.000000,NaN,27143.297500,NaN,210253.000000,3.000000


# PIPELINE BÁSICA

### HACER LOS MERGERS

In [208]:
# Partir de claims (cada fila = 1 siniestro)

# 1. Cruzamos con policies solo por policy_id (eliminamos customer_id para evitar duplicado)
df = df_claims.merge(
    df_policies.drop(columns=['customer_id']), 
    on="policy_id", 
    how="left"
)

# 2. Cruzamos con customers por customer_id
df = df.merge(df_customers, on="customer_id", how="left")

# 3. Cruzamos con vehicles por vehicle_id (eliminamos policy_id para evitar duplicado)
df = df.merge(
    df_vehicles.drop(columns=['policy_id']), 
    on="vehicle_id",  
    how="left"
)

print(df.shape)

(12267, 45)


In [209]:
df

,claim_id,policy_id,customer_id,vehicle_id,agent_id,accident_datetime,claim_datetime,fault,accident_area,accident_description,accident_latitude,accident_longitude,police_report_filed,witness_present,number_of_supplements,claimed_amount_eur,repair_workshop,policy_type,base_policy,deductible,annual_premium_eur,policy_start_date,policy_end_date,past_number_of_claims,number_of_cars,full_name,sex,marital_status,date_of_birth,email,phone,address,city,province,postal_code,last_address_change_date,license_plate,make,model,manufacture_year,vehicle_category,purchase_price_eur,color,odometer_km,driver_rating
0,CLM_00002,POL_07423,CUS_01111,VEH_01155,AGT_00118,2025-11-02 16:33:00,2025-11-28 06:33:00,Policy Holder,Urban,flooding damage,41.104921,-1.472463,Yes,Unknown,7,1198.91,Garage Ramírez - Murcia,Sedan - All Perils,All Perils,300,659.45,2024-05-09,2025-05-09,none,6,Teresa Flores Blanco,Female,Married,1941-07-22,teresa.flores323@hotmail.com,+34 619 772 219,154 Olive Rd,Albacete,Albacete,7046,NaN,1844 XYN,Lexus,NX,2019.0,Coupe,32542.84,Negro,54998.0,3.0
1,CLM_00003,POL_01086,CUS_01750,VEH_00648,AGT_00027,2024-05-12 22:36:00,2024-06-23 15:36:00,Policy Holder,Urban,glass breakage,37.332902,-1.063439,No,Yes,7,11848.24,Carrocerías Díaz - Coruña,Sport - All Perils,All Perils,300,293.08,2021-11-11,2022-11-11,1,3,Paula González Gómez,Female,Married,2004-10-04,paula.gonzalez914@gmail.com,+34 698 603 915,451 Oak St,Fuenlabrada,Madrid,13010,2025-01-07,2010 LCJ,Suzuki,Swift,2007.0,Sedan,57394.78,Blanco,68453.0,2.0
2,CLM_00004,POL_06373,CUS_06806,VEH_00899,AGT_00051,2025-07-09 22:55:00,2025-08-03 02:55:00,Policy Holder,Urban,bicycle involved,36.944930,-6.714446,Yes,No,2,1116.78,Carrocerías Díaz - Vigo,Sport - Collision,Collision,500,1098.78,2024-06-24,2025-06-24,1,2,Miguel Hernández Díaz,Male,Married,1943-12-11,miguel.hernandez440@yahoo.es,+34 655 140 464,194 Spruce Dr,Granada,Granada,7790,NaN,5217 ZCT,Opel,Corsa,2015.0,Coupe,11722.73,Negro,43399.0,1.0
3,CLM_00005,POL_04543,CUS_04165,VEH_05968,AGT_00062,2023-11-08 20:14:00,2023-11-20 17:14:00,Policy Holder,Urban,multi-vehicle pileup,42.466545,2.320600,Yes,Yes,1,2246.88,Carrocerías Ruiz - Valencia,Utility - All Perils,All Perils,400,779.88,2022-11-08,2023-11-08,3,1,Francisco Romero Núñez,Male,Widow,1983-09-02,francisco.romero761@mail.com,+34 619 475 863,345 Willow Way,Logroño,La Rioja,26498,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CLM_00006,POL_04676,CUS_07070,VEH_06082,AGT_00076,2025-08-21 12:39:00,2025-09-30 18:39:00,Policy Holder,Urban,falling object,39.341821,-2.414222,No,Yes,6,449.41,Carrocerías Gómez - Valencia,Utility - All Perils,All Perils,300,529.95,2023-07-02,2024-07-01,1,6,Silvia Vázquez Rodríguez,Female,Divorced,1977-11-14,silvia.vazquez100@mail.com,+34 617 803 407,335 Oak Way,Coruña,A Coruña,25056,NaN,8590 FGD,Chevrolet,Malibu,2013.0,Coupe,15236.23,Negro,86004.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12262,CLM_15412,POL_07393,CUS_02209,VEH_07329,AGT_00123,2024-12-17 11:01:00,2024-12-26 16:01:00,Policy Holder,Suburban,pedestrian involved,39.549568,0.227634,Yes,No,0,960.23,Mecánica Rubio - Málaga,Sport - All Perils,All Perils,500,800.01,2022-09-17,2023-09-17,none,4,Ana Ramos Reyes,Female,Married,1955-12-02,ana.ramos856@yahoo.es,+34 697 384 886,244 Pine Ln,Barcelona,Barcelona,52526,2024-09-10,6942 KZS,Honda,Fit,2006.0,Sedan,25053.98,Plata,147988.0,4.0
12263,CLM_15413,POL_00752,CUS_05111,VEH_02884,AGT_00105,2023-06-24 02:20:00,2023-07-04 19:20:00,Policy Holder,Urban,theft attempt,38.652884,-3.561560,No,No,2,1843.21,Taller Domínguez - Alicante,Sedan - Collision,Collision,300,534.85,2020-02-24,2021-02-23,3,3,Raúl Gil Jiménez,Male,Single,2005-07-13,raul.gil207@gmail.com,+34 661 953 759,361 Maple Dr,Bilbao,Vizcaya,27669,2024-10-19,9627 RXP,Ford,Focus,2021.0,Sedan,36572.27,Blanco,256229.0,4.0
12264,CLM_15414,POL_04232,CUS_01224,VEH_03091,AGT_00042,

In [210]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12267 entries, 0 to 12266
Data columns (total 45 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   claim_id                  12267 non-null  object 
 1   policy_id                 12267 non-null  object 
 2   customer_id               12267 non-null  object 
 3   vehicle_id                12267 non-null  object 
 4   agent_id                  12267 non-null  object 
 5   accident_datetime         12267 non-null  object 
 6   claim_datetime            12267 non-null  object 
 7   fault                     12267 non-null  object 
 8   accident_area             12267 non-null  object 
 9   accident_description      12267 non-null  object 
 10  accident_latitude         12267 non-null  float64
 11  accident_longitude        12267 non-null  float64
 12  police_report_filed       12267 non-null  object 
 13  witness_present           12267 non-null  object 
 14  number

In [211]:
df_target = pd.read_csv("..\\data\\df_target.csv")

# Añadir fraud_flag al dataframe principal por claim_id
df = df.merge(df_target, on="claim_id", how="left")

print(df.shape)          # → (12267, 46)
print(df['fraud_flag'].value_counts())
print(f"Tasa de fraude: {df['fraud_flag'].mean():.2%}")

(12267, 46)
fraud_flag
0    11389
1      878
Name: count, dtype: int64
Tasa de fraude: 7.16%


In [212]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12267 entries, 0 to 12266
Data columns (total 46 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   claim_id                  12267 non-null  object 
 1   policy_id                 12267 non-null  object 
 2   customer_id               12267 non-null  object 
 3   vehicle_id                12267 non-null  object 
 4   agent_id                  12267 non-null  object 
 5   accident_datetime         12267 non-null  object 
 6   claim_datetime            12267 non-null  object 
 7   fault                     12267 non-null  object 
 8   accident_area             12267 non-null  object 
 9   accident_description      12267 non-null  object 
 10  accident_latitude         12267 non-null  float64
 11  accident_longitude        12267 non-null  float64
 12  police_report_filed       12267 non-null  object 
 13  witness_present           12267 non-null  object 
 14  number

In [213]:
# ¿Cuántos vehicle_ids de claims no están en vehicles?
ids_en_claims = set(df_claims['vehicle_id'])
ids_en_vehicles = set(df_vehicles['vehicle_id'])
sin_match = ids_en_claims - ids_en_vehicles
print(f"vehicle_ids sin match: {len(sin_match)}")

vehicle_ids sin match: 1347


In [214]:
# Comprobación de consistencia
check = df_claims.merge(df_policies[['policy_id', 'customer_id']], on='policy_id', suffixes=('_claim', '_policy'))
inconsistentes = check[check['customer_id_claim'] != check['customer_id_policy']]
print(f"Siniestros con customer_id inconsistente: {len(inconsistentes)}")

Siniestros con customer_id inconsistente: 0


In [215]:
# ¿Los siniestros sin datos de vehículo tienen más fraude?
df['vehicle_missing'] = df['make'].isna().astype(int)
print(df.groupby('vehicle_missing')['fraud_flag'].mean().rename({0: 'con vehículo', 1: 'sin vehículo'}))

vehicle_missing
con vehículo    0.072966
sin vehículo    0.066162
Name: fraud_flag, dtype: float64


In [216]:
def load_and_merge_data():
    df_claims    = pd.read_csv("../data/claims.csv")
    df_policies  = pd.read_csv("../data/policies.csv")
    df_customers = pd.read_csv("../data/customers.csv")
    df_vehicles  = pd.read_csv("../data/vehicles.csv")
    df_target    = pd.read_csv("../data/df_target.csv")

    df = df_claims.merge(df_policies.drop(columns=["customer_id"]), on="policy_id", how="left")
    df = df.merge(df_customers, on="customer_id", how="left")
    df = df.merge(df_vehicles.drop(columns=["policy_id"]), on="vehicle_id", how="left")
    df = df.merge(df_target, on="claim_id", how="left")

    return df


def base_transform_names_out(trf, names_in):
  return []

def base_transform(X):
  X[[
      "accident_latitude", "accident_longitude", "number_of_supplements", "claimed_amount_eur",
      "deductible", "annual_premium_eur", "number_of_cars", "manufacture_year",
      "purchase_price_eur", "odometer_km", "driver_rating", "vehicle_missing"
  ]] = (
      X[[
          "accident_latitude", "accident_longitude", "number_of_supplements", "claimed_amount_eur",
          "deductible", "annual_premium_eur", "number_of_cars", "manufacture_year",
          "purchase_price_eur", "odometer_km", "driver_rating", "vehicle_missing"
      ]]
      .replace(r'^\s*$', np.nan, regex=True)
      .apply(pd.to_numeric, errors='coerce')
  )
  X[[
      "fault", "accident_area", "accident_description", "police_report_filed", "witness_present",
      "policy_type", "base_policy", "past_number_of_claims", "sex", "marital_status", "city",
      "province", "make", "vehicle_category", "color"
  ]] = X[[
      "fault", "accident_area", "accident_description", "police_report_filed", "witness_present",
      "policy_type", "base_policy", "past_number_of_claims", "sex", "marital_status", "city",
      "province", "make", "vehicle_category", "color"
  ]].astype("category")
  return X

def get_base_preprocess_pipeline():

  trf = FunctionTransformer(base_transform, feature_names_out=base_transform_names_out)

  trf_columns = []

  drop_columns = [
      "claim_id", "policy_id", "customer_id", "vehicle_id", "agent_id", "accident_datetime",
      "claim_datetime", "repair_workshop", "policy_start_date", "policy_end_date", "full_name",
      "date_of_birth", "email", "phone", "address", "last_address_change_date", "license_plate", "model",
      "postal_code", "fraud_flag"
  ]
  base_pandas_trf = ColumnTransformer(
      transformers=[
      ("trf", trf, trf_columns),
      ("drop", "drop", drop_columns)
      ],
      remainder="passthrough",
      verbose_feature_names_out=True
  )

  cat_transformer = Pipeline(steps=[
      ('imputer', SimpleImputer(strategy='most_frequent')),
      ('onehot', OneHotEncoder(drop="first", sparse_output=False, handle_unknown='ignore'))
  ])

  num_transformer = Pipeline(steps=[
      ('imputer', SimpleImputer(strategy='median')),
      ('scaler', StandardScaler())
  ])

  transformer = ColumnTransformer(
      transformers=[
          ('num', num_transformer, selector(dtype_exclude="category")),
          ('cat', cat_transformer, selector(dtype_include="category"))
      ])

  preprocess_pipeline = Pipeline(steps=[
      ('trf', base_pandas_trf),
      ('transformer', transformer),
      ('variance_threshold', VarianceThreshold(threshold=0))
      ])

  return preprocess_pipeline

In [217]:
# Comprobacion breve: num + cat + drop vs columnas totales de df
num_cols = [
    "accident_latitude", "accident_longitude", "number_of_supplements", "claimed_amount_eur",
    "deductible", "annual_premium_eur", "number_of_cars", "manufacture_year",
    "purchase_price_eur", "odometer_km", "driver_rating", "vehicle_missing"
]

cat_cols = [
    "fault", "accident_area", "accident_description", "police_report_filed", "witness_present",
    "policy_type", "base_policy", "past_number_of_claims", "sex", "marital_status", "city",
    "province", "make", "vehicle_category", "color"
]

drop_cols = [
    "claim_id", "policy_id", "customer_id", "vehicle_id", "agent_id", "accident_datetime",
    "claim_datetime", "repair_workshop", "policy_start_date", "policy_end_date", "full_name",
    "date_of_birth", "email", "phone", "address", "last_address_change_date", "license_plate", "model",
    "postal_code", "fraud_flag"
]

n_df = df.shape[1]
n_num = len(num_cols)
n_cat = len(cat_cols)
n_drop = len(drop_cols)

union_cols = set(num_cols) | set(cat_cols) | set(drop_cols)
intersections = {
    "num_cat": set(num_cols) & set(cat_cols),
    "num_drop": set(num_cols) & set(drop_cols),
    "cat_drop": set(cat_cols) & set(drop_cols),
}

print(f"df columnas: {n_df}")
print(f"num: {n_num}, cat: {n_cat}, drop: {n_drop}, suma: {n_num + n_cat + n_drop}")
print(f"union unica: {len(union_cols)}")
print(f"coincide_con_df: {len(union_cols) == n_df}")

for k, v in intersections.items():
    print(f"{k}: {len(v)}")

faltan_en_listas = sorted(set(df.columns) - union_cols)
extra_en_listas = sorted(union_cols - set(df.columns))
print("faltan_en_listas:", faltan_en_listas)
print("extra_en_listas:", extra_en_listas)

df columnas: 47
num: 12, cat: 15, drop: 20, suma: 47
union unica: 47
coincide_con_df: True
num_cat: 0
num_drop: 0
cat_drop: 0
faltan_en_listas: []
extra_en_listas: []


# Ideas para feature
- claims.csv
1. Sacar la provincia del accidente con las columnas:
`accident_latitude` — Latitud de la ubicación del accidente (coordenadas en España).
`accident_longitude` — Longitud de la ubicación del accidente.
2. Sacar la provincia del taller
`repair_workshop`
- pólices.csv
1. Hacer una variable de si le queda poco tiempo con las columnas
`policy_start_date` — Fecha de inicio de la póliza. Formato: `YYYY-MM-DD`.
`policy_end_date` — Fecha de fin de la póliza. Siempre 365 días después de `policy_start_date`. Formato: `YYYY-MM-DD`.



Pasar la columa de past_number_of_claims a numérica
